In [1]:
import pandas as pd
data = pd.read_json("games.json")
data = data.transpose()

In [2]:
# View data
data

,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,peak_ccu,tags,discount
20200,Galactic Bowling,"Oct 21, 2008",0,19.99,0,Galactic Bowling is an exaggerated and stylize...,Galactic Bowling is an exaggerated and stylize...,Galactic Bowling is an exaggerated and stylize...,,https://cdn.akamai.steamstatic.com/steam/apps/...,...,6,11,0 - 20000,0,0,0,0,0,"{'Indie': 22, 'Casual': 21, 'Sports': 21, 'Bow...",NaN
655370,Train Bandit,"Oct 12, 2017",0,0.99,0,THE LAW!! Looks to be a showdown atop a train....,THE LAW!! Looks to be a showdown atop a train....,THE LAW!! Looks to be a showdown atop a train....,,https://cdn.akamai.steamstatic.com/steam/apps/...,...,53,5,0 - 20000,0,0,0,0,0,"{'Indie': 109, 'Action': 103, 'Pixel Graphics'...",NaN
1732930,Jolt Project,"Nov 17, 2021",0,4.99,0,Jolt Project: The army now has a new robotics ...,Jolt Project: The army now has a new robotics ...,"Shoot vehicles, blow enemies with a special at...",,https://cdn.akamai.steamstatic.com/steam/apps/...,...,0,0,0 - 20000,0,0,0,0,0,[],NaN
1355720,Henosis™,"Jul 23, 2020",0,5.99,0,HENOSIS™ is a mysterious 2D Platform Puzzler w...,HENOSIS™ is a mysterious 2D Platform Puzzler w...,HENOSIS™ is a mysterious 2D Platform Puzzler w...,,https://cdn.akamai.steamstatic.com/steam/apps/...,...,3,0,0 - 20000,0,0,0,0,0,"{'2D Platformer': 161, 'Atmospheric': 154, 'Su...",NaN
1139950,Two Weeks in Painland,"Feb 3, 2020",0,0.0,0,ABOUT THE GAME Play as a hacker who has arrang...,ABOUT THE GAME Play as a hacker who has arrang...,Two Weeks in Painland is a story-driven game a...,,https://cdn.akamai.steamstatic.com/steam/apps/...,...,50,8,0 - 20000,0,0,0,0,0,"{'Indie': 42, 'Adventure': 41, 'Nudity': 22, '...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3600970,Paragon Of Time,"Apr 10, 2025",0,2.99,0,"You stand at the edge of time, trying to save ...","You stand at the edge of time, trying to save ...",Crush the darkness that is corrupting time its...,,https://shared.akamai.steamstatic.com/store_it...,...,5,0,0 - 20000,0,0,0,0,0,"{'Action Roguelike': 296, 'Bullet Hell': 290, ...",0
3543710,A Few Days With : Hazel,"Apr 11, 2025",0,2.69,0,"Join Hazel, an attractive young lady, and enjo...","Join Hazel, an attractive young lady, and enjo...",Embark on a sensual butterfly-hunting adventur...,,https://shared.akamai.steamstatic.com/store_it...,...,0,0,0 - 20000,0,0,0,0,0,[],10
3265370,MosGhost,"Apr 1, 2025",0,7.99,0,Check out ADD TO WISHLIST About the Game Story...,Story : Andrei moved to Moscow for work and re...,"Having moved to Moscow, Andrei rents a cheap a...",,https://shared.akamai.steamstatic.com/store_it...,...,24,12,0 - 20000,0,0,0,0,0,"{'Simulation': 70, 'Walking Simulator': 44, 'I...",0
3423620,AccuBow VR,"Mar 11, 2025",0,0.0,0,AccuBow VR: Master Realistic Archery in Immers...,AccuBow VR: Master Realistic Archery in Immers...,Take your archery training to the next level a...,,https://shared.akamai.steamstatic.com/store_it...,...,0,0,0 - 0,0,0,0,0,0,[],0


In [3]:
# Lets take a look at the tags. Each value in the tags column is a dict with each tag and some other value
from collections import defaultdict
all_tags = defaultdict(int)
error_count = 0  # Some games have no tags, this counts those objects

for idx, row in data.iterrows():
    try:
        for key, val in row["tags"].items():
            all_tags[key] += val
    except AttributeError:
        error_count += 1

print(f"Number of tags: {len(all_tags)}")
print(f"ERROR COUNT (Number of games with no tags): {error_count}")
print("\nBelow are the tags, sorted in descending order by the totals of the numbers assigned to each tag.")
print("I would assume the number assigned to tags is the importance? Something like the number of users that suggested the tag.")

sorted_items = sorted(all_tags.items(), key=lambda x: x[1], reverse=True)
for obj in sorted_items:
    print(f"{obj[0]:<30} | {obj[1]:,}")

Number of tags: 453
ERROR COUNT (Number of games with no tags): 37423

Below are the tags, sorted in descending order by the totals of the numbers assigned to each tag.
I would assume the number assigned to tags is the importance? Something like the number of users that suggested the tag.
Action                         | 4,137,255
Adventure                      | 3,616,025
Singleplayer                   | 3,282,316
Casual                         | 3,032,026
Indie                          | 2,828,392
2D                             | 2,298,267
Strategy                       | 2,046,396
RPG                            | 1,978,673
Simulation                     | 1,962,669
Exploration                    | 1,798,982
3D                             | 1,762,758
Atmospheric                    | 1,750,405
Multiplayer                    | 1,493,343
First-Person                   | 1,491,734
Puzzle                         | 1,483,708
Early Access                   | 1,482,198
Story Rich            

In [4]:
# Let's see if we can judge similarity with euclidean distance.
# First we need to extract relevant data into a better format.
# Each object will have its AppID, name, and then columns for each tag

# Creating column names
COLUMN_NAMES = ['AppID', 'Name']  # non-tag columns
alphabetized_tags = sorted(all_tags.keys(), key=lambda x: x[0].upper(), reverse=False)
for tag in alphabetized_tags:
    COLUMN_NAMES.append(f"tag_{tag}")
features_df = pd.DataFrame(columns=COLUMN_NAMES)

# Populate columns with data
results = []
for idx, row in data.iterrows():
    new_row = defaultdict(lambda: 0)
    
    # Populate constant fields
    new_row['AppID'] = idx
    new_row['Name'] = row['name']
    new_row['Num_Reviews'] = row['positive'] + row['negative']

    # Populate tags
    try:
        for key, val in row["tags"].items():
            key_name = f"tag_{key}"
            new_row[key_name] = val
    except AttributeError:  # No tags exist for this game if exception is thrown
        pass

    results.append(dict(new_row))

# Final construction of dataframe
features_df = pd.DataFrame(results, columns=COLUMN_NAMES).set_index('AppID')
features_df = features_df.fillna(0)

# Output
print(features_df)
features_df.to_csv("tags.csv")

                            Name  tag_1990's  tag_1980s  tag_2D  \
AppID                                                             
20200           Galactic Bowling         0.0        0.0     0.0   
655370              Train Bandit         0.0        0.0    97.0   
1732930             Jolt Project         0.0        0.0     0.0   
1355720                 Henosis™         0.0        0.0    56.0   
1139950    Two Weeks in Painland         0.0        0.0     0.0   
...                          ...         ...        ...     ...   
3600970          Paragon Of Time         0.0        0.0   258.0   
3543710  A Few Days With : Hazel         0.0        0.0     0.0   
3265370                 MosGhost         0.0        0.0     0.0   
3423620               AccuBow VR         0.0        0.0     0.0   
3183790   Defense Of Fort Burton         0.0        0.0     0.0   

         tag_2D Platformer  tag_2.5D  tag_2D Fighter  tag_3D  tag_3D Vision  \
AppID                                            

In [12]:
# Now we define a similarity finding function
from math import sqrt, fabs

def calc_euclid_similarity(row1, row2):
    # Calculate euclidean distance
    acc = 0
    for col, _ in row1.items():
        if "tag" in col:
            acc += (row1[col] - row2[col]) ** 2
    return sqrt(acc)

def calc_manhattan_similarity(row1, row2):
    # Calculate euclidean distance
    acc = 0
    for col, _ in row1.items():
        if "tag" in col:
            acc += fabs(row1[col] - row2[col])
    return acc
            
def similar_to(df, id, dist_func=calc_euclid_similarity):
    # Find row most similar to given AppID
    main_row = df.loc[id]

    # Establish defaults
    min_row = df.iloc[0]
    min_score = dist_func(main_row, min_row) + 99999999999

    for idx, other_row in df.iterrows():
        score = dist_func(main_row, other_row)
        # Check if this row is most similar found so far
        if min_score > score and score != 0:
            min_row = other_row
            min_score = score

            print(f"New row: {idx} ({min_row['Name']} | {min_score})") # debug

    return min_row

In [14]:
print("Using Euclidean distance...")
similar_col = similar_to(features_df, 20200)
print("\nResults:")
print(similar_col)

print("\nUsing Manhattan distance...")
similar_col = similar_to(features_df, 20200, dist_func=calc_manhattan_similarity)
print("\nResults:")
print(similar_col)

Using Euclidean distance...
New row: 655370 (Train Bandit | 315.9984177175576)
New row: 1732930 (Jolt Project | 37.44329045369811)
New row: 825930 (Royal Battleships | 21.840329667841555)
New row: 1028340 (Spare Teeth VR | 18.894443627691185)
New row: 1156200 (VirtuaLiron - Immersive YOGA practice | 18.49324200890693)
New row: 497480 (GravPool | 11.74734012447073)
New row: 658880 (Jammerball | 6.082762530298219)

Results:
Name                 Jammerball
tag_1990's                  0.0
tag_1980s                   0.0
tag_2D                      0.0
tag_2D Platformer           0.0
                        ...    
tag_Warhammer 40K           0.0
tag_Well-Written            0.0
tag_Wrestling               0.0
tag_Wholesome               0.0
tag_Zombies                 0.0
Name: 658880, Length: 454, dtype: object

Using Manhattan distance...
New row: 655370 (Train Bandit | 1351.0)
New row: 1732930 (Jolt Project | 70.0)
New row: 825930 (Royal Battleships | 27.0)
New row: 497480 (GravPool | 18

In [10]:
# Lets see if this is legit
print(data.loc[20200]["tags"])
print(data.loc[658880]["tags"])
# Though the games themselves aren't very similar. One is a dodgeball game and the other is bowling.
# Note Jammerball has many more tags on its actual steam page: Arcade, 3D, PvP, etc. that are not included in this dataset.

{'Indie': 22, 'Casual': 21, 'Sports': 21, 'Bowling': 6}
{'Indie': 21, 'Casual': 21, 'Sports': 21}


In [13]:
# Lets try TF2, a more familiar game than Galactic Bowling

print("Using Euclidean distance...")
similar_col = similar_to(features_df, 440)
print("\nResults:")
print(similar_col)

print("\nUsing Manhattan distance...")
similar_col = similar_to(features_df, 440, dist_func=calc_manhattan_similarity)
print("\nResults:")
print(similar_col)

Using Euclidean distance...
New row: 20200 (Galactic Bowling | 94872.2671912082)
New row: 655370 (Train Bandit | 94851.19192187308)
New row: 1469160 (Wartune Reborn | 94765.57413955766)
New row: 1192900 (IRON REBELLION | 94635.72967965112)
New row: 552520 (Far Cry® 5 | 94187.06960087462)
New row: 1172470 (Apex Legends™ | 93049.21057161098)
New row: 488821 (Tom Clancy's Rainbow Six® Siege | 90894.27209125996)
New row: 488824 (Tom Clancy's Rainbow Six® Siege | 90892.95509554082)
New row: 359550 (Tom Clancy's Rainbow Six® Siege | 84922.35675603923)
New row: 570 (Dota 2 | 72888.10536020264)
New row: 444090 (Paladins® | 66497.09515460055)

Results:
Name                 Paladins®
tag_1990's                 0.0
tag_1980s                  0.0
tag_2D                     0.0
tag_2D Platformer          0.0
                       ...    
tag_Warhammer 40K          0.0
tag_Well-Written           0.0
tag_Wrestling              0.0
tag_Wholesome              0.0
tag_Zombies                0.0
Name: 4

In [22]:
# Lets test this too
print(data.loc[440]["tags"])
print(data.loc[444090]["tags"])

{'Free to Play': 61536, 'Hero Shooter': 60556, 'Multiplayer': 21875, 'FPS': 14723, 'Shooter': 12009, 'Action': 11510, 'Class-Based': 9112, 'Team-Based': 8741, 'Funny': 8576, 'First-Person': 7341, 'Online Co-Op': 6833, 'Competitive': 6685, 'Cartoony': 6679, 'Trading': 6467, 'Co-op': 5403, 'Comedy': 4849, 'Robots': 4351, 'Tactical': 3700, 'Cartoon': 3626, 'Crafting': 3450}
{'Hero Shooter': 50267, 'Free to Play': 5229, 'Multiplayer': 3228, 'FPS': 3102, 'Shooter': 2714, 'Action': 2195, 'Team-Based': 2152, 'First-Person': 1744, 'PvP': 1700, 'MOBA': 1606, 'Online Co-Op': 1494, 'Fantasy': 1173, 'Strategy': 1055, 'Co-op': 1041, 'Early Access': 1003, 'Massively Multiplayer': 862, 'Funny': 733, 'Adventure': 540, 'Survival': 527, 'Anime': 449}


In [26]:
# This seems to take a long time. Let's try and remove some games that are irrelevant.
# My approach will be to sum up the tag numbers and remove any under a certain threshold
MINIMUM_TAG = 1000

def should_keep_row(row):
    result = 0
    for col, val in row.items():
        if "tag" in col:
            result += val
    return result > MINIMUM_TAG

# Apply function to each row, get boolean mask
mask = features_df.apply(should_keep_row, axis=1)
reduced_features_df = features_df[mask]

print(f"Original:{len(features_df):>8} rows")
print(f"Reduced: {len(reduced_features_df):>8} rows")
print(f"Subtracted: {len(features_df) - len(reduced_features_df):5} rows")

Original:  111452 rows
Reduced:    29157 rows
Subtracted: 82295 rows


In [27]:
# Lets try the similarity on the reduced data set
print("Using Euclidean distance...")
similar_col = similar_to(reduced_features_df, 440)
print("\nResults:")
print(similar_col)

print("\nUsing Manhattan distance...")
similar_col = similar_to(reduced_features_df, 440, dist_func=calc_manhattan_similarity)
print("\nResults:")
print(similar_col)

Using Euclidean distance...
New row: 655370 (Train Bandit | 94851.19192187308)
New row: 1469160 (Wartune Reborn | 94765.57413955766)
New row: 1192900 (IRON REBELLION | 94635.72967965112)
New row: 552520 (Far Cry® 5 | 94187.06960087462)
New row: 1172470 (Apex Legends™ | 93049.21057161098)
New row: 488821 (Tom Clancy's Rainbow Six® Siege | 90894.27209125996)
New row: 488824 (Tom Clancy's Rainbow Six® Siege | 90892.95509554082)
New row: 359550 (Tom Clancy's Rainbow Six® Siege | 84922.35675603923)
New row: 570 (Dota 2 | 72888.10536020264)
New row: 444090 (Paladins® | 66497.09515460055)

Results:
Name                 Paladins®
tag_1990's                 0.0
tag_1980s                  0.0
tag_2D                     0.0
tag_2D Platformer          0.0
                       ...    
tag_Warhammer 40K          0.0
tag_Well-Written           0.0
tag_Wrestling              0.0
tag_Wholesome              0.0
tag_Zombies                0.0
Name: 444090, Length: 454, dtype: object

Using Manhattan di

In [31]:
# Testing speedup of reduced dataset
from timeit import default_timer as timer

appid = 1055540

full_start = timer()
print("Using Euclidean distance on full dataset")
similar_col = similar_to(features_df, appid)
full_end = timer()

reduced_start = timer()
print("\nUsing Euclidean distance on reduced dataset")
similar_col = similar_to(reduced_features_df, appid)
reduced_end = timer()

print(f"\nFull time: {full_end - full_start}\nReduced time: {reduced_end - reduced_start}")

Using Euclidean distance on full dataset
New row: 20200 (Galactic Bowling | 391.2531150035741)
New row: 1139950 (Two Weeks in Painland | 376.84877603622385)
New row: 346560 (Hero of the Kingdom II | 361.6517662061116)
New row: 1292520 (Crimson Spires | 359.5427651893443)
New row: 1467740 (Pocket Plants | 347.64637205068027)
New row: 266490 (Lili: Child of Geos - Complete Edition | 333.9266386498687)
New row: 244710 (Shelter | 298.53642993778834)
New row: 555150 (The First Tree | 298.41079068961295)
New row: 1335680 (The Kind Camomille | 275.90215657004205)
New row: 2600720 (Cloudy Valley | 244.8999795835026)

Using Euclidean distance on reduced dataset
New row: 655370 (Train Bandit | 437.15900997234405)
New row: 955490 (Terror In The Atomic Desert | 381.90705675595996)
New row: 559210 (Rakuen | 374.1497026592431)
New row: 1192500 (GraFi Christmas | 370.7155243579637)
New row: 1991470 (Cold Land | 356.7590223105787)
New row: 463760 (The Beggar's Ride | 338.1819037145542)
New row: 244710

In [32]:
# The speedup is great, but it's odd that the reduced dataset kept suggested a less popular game. Let's investigate
print("Cloudy Valley")
print(data.loc[2600720]["tags"])

print("Skolly's Adventure")
print(data.loc[2573930]["tags"])

Cloudy Valley
{'Adventure': 88, 'Exploration': 81, 'Cute': 77, 'Indie': 74, 'Relaxing': 70, 'Casual': 64, 'Open World': 59, 'Nature': 57, 'Singleplayer': 52, 'Colorful': 49, 'Walking Simulator': 47, 'Female Protagonist': 45, 'Ambient': 43, 'Wholesome': 41, 'Third Person': 29, 'Family Friendly': 27}
Skolly's Adventure
{'Adventure': 95, 'Exploration': 89, 'Cute': 86, 'Open World': 83, 'Short': 80, '3D Platformer': 75, 'Relaxing': 72, 'Casual': 69, 'Puzzle': 61, 'Nature': 59, 'Puzzle-Platformer': 57, 'Platformer': 55, 'Atmospheric': 53, 'Indie': 51, 'Family Friendly': 31, 'Third Person': 29, 'Singleplayer': 27, '3D': 25, 'Colorful': 23, 'Collectathon': 21}


In [ ]:
# It looks like it might overweight the significance of games with many tags applied.
# Lets try and use the number of reviews
MINIMUM_TAG = 1000


def should_keep_row(row):
    result = 0
    for col, val in row.items():
        if "tag" in col:
            result += val
    return result > MINIMUM_TAG

# Apply function to each row, get boolean mask
mask = features_df.apply(should_keep_row, axis=1)
reduced_features_df = features_df[mask]

print(f"Original:{len(features_df):>8} rows")
print(f"Reduced: {len(reduced_features_df):>8} rows")
print(f"Subtracted: {len(features_df) - len(reduced_features_df):5} rows")